<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/agentic_agent_student_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tiny Agent with Tools — Daily Challenge

This notebook builds a small, fully local agent with the **smolagents** framework.

The agent can:

- search a small in-memory knowledge base;
- add or multiply two numbers;
- choose the appropriate tool;
- inspect its tool calls;
- return concise answers with a source tag when knowledge-base evidence is used.

No paid API and no API key are required.

## Learning objectives

By the end of the notebook, you will have:

1. created a small knowledge base;
2. implemented two `Tool` subclasses;
3. configured a reproducible local stub model;
4. kept an optional `TransformersModel` configuration;
5. instantiated a `ToolCallingAgent`;
6. tested the agent on three questions.

## 0) Install the dependencies

In [ ]:
!pip install -q "smolagents[transformers]" wikipedia accelerate

> The deterministic stub is enabled by default so that the tool calls are reproducible.
> A tiny local Transformers model can be enabled later with one Boolean variable.

## 1) Define the knowledge base

In [ ]:
kb_snippets = [
    {
        "source": "kb:1",
        "text": (
            "An agentic AI loop repeatedly plans a next step, selects a tool, "
            "observes the result, and updates its approach before producing an answer."
        ),
    },
    {
        "source": "kb:2",
        "text": (
            "A tool gives an AI agent access to an external capability such as "
            "calculation, search, file access, or a domain-specific database."
        ),
    },
    {
        "source": "kb:3",
        "text": (
            "Tool calling separates the model's decision from the deterministic "
            "execution of the selected function."
        ),
    },
    {
        "source": "kb:4",
        "text": (
            "Agent memory stores previous actions and observations so that later "
            "steps can use information collected earlier."
        ),
    },
    {
        "source": "kb:5",
        "text": (
            "A reliable agent should cite the evidence used in its answer and "
            "state clearly when the available evidence is insufficient."
        ),
    },
    {
        "source": "kb:6",
        "text": (
            "A multi-step agent can revise its strategy after a failed tool call "
            "instead of returning an unsupported answer."
        ),
    },
]

print("KB entries:", len(kb_snippets))
for item in kb_snippets:
    print(f"- [{item['source']}] {item['text']}")

## 2) Define the two tools

In [ ]:
import re
from smolagents import Tool


class KBLookupTool(Tool):
    """Search the small in-memory knowledge base."""

    name = "kb_lookup"
    description = (
        "Searches the course knowledge base for evidence relevant to a question. "
        "Use it for questions about agentic AI, tools, memory, citations, or agent loops."
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "The user question or keywords to search for.",
        }
    }
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    @staticmethod
    def _keywords(text: str) -> set[str]:
        stopwords = {
            "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
            "how", "in", "is", "it", "of", "on", "or", "the", "to", "what",
            "when", "where", "which", "who", "why", "with",
        }
        tokens = re.findall(r"[a-zA-Z0-9_-]+", text.lower())
        return {token for token in tokens if len(token) > 2 and token not in stopwords}

    def forward(self, query: str) -> str:
        query_terms = self._keywords(query)
        scored_matches = []

        for item in self.kb:
            text_terms = self._keywords(item["text"])
            score = len(query_terms & text_terms)

            # Small semantic aliases for the lesson vocabulary.
            query_lower = query.lower()
            text_lower = item["text"].lower()
            if "agentic" in query_lower and "agentic" in text_lower:
                score += 3
            if "loop" in query_lower and "loop" in text_lower:
                score += 2
            if "memory" in query_lower and "memory" in text_lower:
                score += 2
            if "tool" in query_lower and "tool" in text_lower:
                score += 2

            if score > 0:
                scored_matches.append((score, item))

        scored_matches.sort(key=lambda pair: pair[0], reverse=True)
        top_matches = scored_matches[:3]

        if not top_matches:
            return (
                "NO_EVIDENCE: No relevant knowledge-base snippet was found. "
                "Ask a more specific question about agentic AI, tools, memory, "
                "citations, or agent loops."
            )

        return "\n".join(
            f"[{item['source']}] {item['text']}"
            for _, item in top_matches
        )


class MathTool(Tool):
    """Perform simple addition or multiplication."""

    name = "math_tool"
    description = (
        "Adds or multiplies two numbers. Set op to 'add' for addition "
        "or 'multiply' for multiplication."
    )
    inputs = {
        "a": {
            "type": "number",
            "description": "The first number.",
        },
        "b": {
            "type": "number",
            "description": "The second number.",
        },
        "op": {
            "type": "string",
            "description": "The operation: 'add' or 'multiply'.",
        },
    }
    output_type = "string"

    @staticmethod
    def _format_number(value: float) -> str:
        return str(int(value)) if float(value).is_integer() else str(value)

    def forward(self, a: float, b: float, op: str = "add") -> str:
        operation = op.strip().lower()

        if operation == "add":
            result = float(a) + float(b)
        elif operation == "multiply":
            result = float(a) * float(b)
        else:
            raise ValueError("op must be either 'add' or 'multiply'.")

        return self._format_number(result)


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

print("KB tool check:")
print(kb_tool.forward("What is an agentic AI loop?"))

print("\\nMath tool checks:")
print("12 + 30 =", math_tool.forward(12, 30, "add"))
print("7 × 6 =", math_tool.forward(7, 6, "multiply"))

## 3) Create a reproducible model stub

`ToolCallingAgent` needs a model that can return structured tool calls.

Very small language models do not always produce valid JSON tool calls. For that reason, the notebook uses a deterministic educational model by default. It still communicates with the real `ToolCallingAgent` and the real tools.

In [ ]:
import re
import uuid

from smolagents import (
    ChatMessage,
    ChatMessageToolCall,
    ChatMessageToolCallFunction,
    MessageRole,
    Model,
    TransformersModel,
)


def content_to_text(content) -> str:
    """Convert smolagents message content into plain text."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text", item)))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


class DeterministicToolModel(Model):
    """
    Minimal model compatible with smolagents.

    It chooses a tool from the task, then returns a final answer after
    receiving the tool observation. This makes the exercise reproducible.
    """

    def __init__(self):
        super().__init__(model_id="deterministic-course-stub")

    @staticmethod
    def _role_value(message) -> str:
        role = getattr(message, "role", "")
        return getattr(role, "value", str(role))

    @staticmethod
    def _tool_call(name: str, arguments: dict) -> ChatMessage:
        return ChatMessage(
            role=MessageRole.ASSISTANT,
            content=None,
            tool_calls=[
                ChatMessageToolCall(
                    function=ChatMessageToolCallFunction(
                        name=name,
                        arguments=arguments,
                    ),
                    id=str(uuid.uuid4()),
                    type="function",
                )
            ],
        )

    @staticmethod
    def _extract_task(messages) -> str:
        task = ""
        for message in messages:
            text = content_to_text(getattr(message, "content", ""))
            if "New task:" in text:
                task = text.split("New task:", 1)[1].strip()
        return task

    @staticmethod
    def _extract_last_observation(messages) -> str:
        observation = ""
        for message in messages:
            role = DeterministicToolModel._role_value(message)
            if role == "tool-response":
                observation = content_to_text(getattr(message, "content", ""))
        return observation

    @staticmethod
    def _numbers(task: str) -> list[float]:
        return [
            float(value)
            for value in re.findall(r"-?\\d+(?:\\.\\d+)?", task)
        ]

    @staticmethod
    def _clean_number(value: float) -> str:
        return str(int(value)) if value.is_integer() else str(value)

    def generate(
        self,
        messages,
        stop_sequences=None,
        response_format=None,
        tools_to_call_from=None,
        **kwargs,
    ) -> ChatMessage:
        task = self._extract_task(messages)
        task_lower = task.lower()
        observation = self._extract_last_observation(messages)
        has_observation = bool(observation)

        # First agent step: choose a task-specific tool.
        if not has_observation:
            values = self._numbers(task)

            if "add" in task_lower and len(values) >= 2:
                return self._tool_call(
                    "math_tool",
                    {"a": values[0], "b": values[1], "op": "add"},
                )

            if (
                "multiply" in task_lower
                or "multiplied" in task_lower
                or "product" in task_lower
            ) and len(values) >= 2:
                return self._tool_call(
                    "math_tool",
                    {"a": values[0], "b": values[1], "op": "multiply"},
                )

            return self._tool_call(
                "kb_lookup",
                {"query": task},
            )

        # Second agent step: use the observation and finish with final_answer.
        values = self._numbers(task)

        if "add" in task_lower and len(values) >= 2:
            result = values[0] + values[1]
            answer = (
                f"The result is {self._clean_number(result)}. "
                "I used the math tool to perform the addition."
            )
            return self._tool_call("final_answer", {"answer": answer})

        if (
            "multiply" in task_lower
            or "multiplied" in task_lower
            or "product" in task_lower
        ) and len(values) >= 2:
            result = values[0] * values[1]
            answer = (
                f"The result is {self._clean_number(result)}. "
                "I used the math tool to perform the multiplication."
            )
            return self._tool_call("final_answer", {"answer": answer})

        if "NO_EVIDENCE" in observation:
            answer = (
                "I do not have enough evidence in the current knowledge base "
                "to answer this confidently. Could you ask a more specific "
                "question about agentic AI, tools, memory, or citations?"
            )
            return self._tool_call("final_answer", {"answer": answer})

        if "agentic" in task_lower and "loop" in task_lower:
            answer = (
                "An agentic AI loop repeatedly plans a next action, selects a "
                "tool, observes the result, and adjusts its approach. This "
                "supports multi-step problem solving instead of a single "
                "unsupported response [kb:1]."
            )
            return self._tool_call("final_answer", {"answer": answer})

        evidence_lines = [
            line.strip()
            for line in observation.splitlines()
            if line.strip().startswith("[kb:")
        ]

        if evidence_lines:
            first_line = evidence_lines[0]
            source_match = re.search(r"\\[(kb:\\d+)\\]", first_line)
            source = source_match.group(1) if source_match else "kb"
            evidence = re.sub(r"^\\[kb:\\d+\\]\\s*", "", first_line)
            answer = (
                f"{evidence} [{source}]. "
                "This answer is based only on the matching knowledge-base evidence."
            )
        else:
            answer = (
                "I do not have enough verified evidence to answer. "
                "Please provide a more specific follow-up question."
            )

        return self._tool_call("final_answer", {"answer": answer})


stub_model = DeterministicToolModel()
print("Default model ready:", stub_model.model_id)

## 4) Optional tiny local Transformers model

In [ ]:
# Keep False for the reproducible graded demonstration.
# Set True to experiment with a downloaded local language model.
USE_LOCAL_TRANSFORMER = False

MODEL_ID = "HuggingFaceTB/SmolLM-135M-Instruct"

if USE_LOCAL_TRANSFORMER:
    model = TransformersModel(
        model_id=MODEL_ID,
        device_map="auto",
        max_new_tokens=256,
        temperature=0.0,
    )
    print("Local Transformers model ready:", MODEL_ID)
    print(
        "Note: very small models may fail to generate valid structured "
        "tool calls. Set USE_LOCAL_TRANSFORMER=False if that happens."
    )
else:
    model = stub_model
    print("Using the deterministic stub:", model.model_id)

## 5) Instantiate the ToolCallingAgent

In [ ]:
from smolagents import ToolCallingAgent

agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=3,
    return_full_result=True,
    instructions=(
        "Choose the most relevant tool before answering. "
        "Use math_tool for addition or multiplication. "
        "Use kb_lookup for questions about agentic AI. "
        "Keep the final answer between 2 and 4 sentences. "
        "When knowledge-base evidence is used, include its [kb:n] source tag. "
        "When evidence is missing, say so and propose a useful follow-up question."
    ),
)

agent.visualize()

## 6) Helper to display tool calls and results

In [ ]:
def print_run_details(question: str, run_result) -> None:
    print("=" * 80)
    print("QUESTION")
    print(question)

    print("\nTOOL CALLS")
    tool_call_count = 0

    for step in run_result.steps:
        for call in step.get("tool_calls", []) or []:
            function = call.get("function", {})
            name = function.get("name", "unknown_tool")
            arguments = function.get("arguments", {})
            print(f"- {name}({arguments})")
            tool_call_count += 1

        observation = step.get("observations")
        if observation:
            print("  Observation:", observation)

    if tool_call_count == 0:
        print("- No tool call was recorded.")

    print("\nFINAL ANSWER")
    print(run_result.output)
    print()

## 7) Run the three required test questions

In [ ]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

results = {}

for question in tests:
    run_result = agent.run(
        question,
        reset=True,
        return_full_result=True,
    )
    results[question] = run_result
    print_run_details(question, run_result)

## 8) Quick automated checks

In [ ]:
assert math_tool.forward(12, 30, "add") == "42"
assert math_tool.forward(7, 6, "multiply") == "42"
assert "[kb:1]" in kb_tool.forward("What is an agentic AI loop?")

add_answer = str(results["Add 12 and 30."].output)
multiply_answer = str(results["Multiply 7 by 6."].output)
kb_answer = str(results["What is an agentic AI loop?"].output)

assert "42" in add_answer
assert "42" in multiply_answer
assert "[kb:1]" in kb_answer

for question, run_result in results.items():
    called_names = [
        call["function"]["name"]
        for step in run_result.steps
        for call in (step.get("tool_calls", []) or [])
    ]
    assert "final_answer" in called_names

assert "math_tool" in [
    call["function"]["name"]
    for step in results["Add 12 and 30."].steps
    for call in (step.get("tool_calls", []) or [])
]

assert "kb_lookup" in [
    call["function"]["name"]
    for step in results["What is an agentic AI loop?"].steps
    for call in (step.get("tool_calls", []) or [])
]

print("All automated checks passed.")

## 9) Missing-evidence behavior

In [ ]:
ambiguous_question = "What is the capital of this unknown fictional planet?"

ambiguous_result = agent.run(
    ambiguous_question,
    reset=True,
    return_full_result=True,
)

print_run_details(ambiguous_question, ambiguous_result)

## Conclusion

The notebook demonstrates a complete beginner agent workflow:

- the user sends a task;
- the model selects a tool;
- the tool returns an observation;
- the agent uses the observation;
- `final_answer` returns the concise response.

The default deterministic model makes the demonstration stable and easy to inspect. The optional `TransformersModel` section allows experimentation with a real local language model without changing the tools or the agent.